### BÚSQUEDA DE HIPERPARÁMETROS

In [ ]:
# =============================================================================
# Configuración de entorno y credenciales (MinIO local)
# =============================================================================

%env AWS_ACCESS_KEY_ID=minio
%env AWS_SECRET_ACCESS_KEY=minio123
%env MLFLOW_S3_ENDPOINT_URL=http://localhost:9000
%env AWS_ENDPOINT_URL_S3=http://localhost:9000

In [ ]:
# =============================================================================
# Librerías
# =============================================================================

import os
import io
import pickle
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import mlflow
import mlflow.pytorch
from mlflow.models.signature import infer_signature
import optuna
from datetime import datetime
import awswrangler as wr
import boto3

# Importar módulos
from model import CNN1DRegressor
from dataset import DiameterDataset, custom_collate_fn
from utils import train_one_epoch, validate, evaluate_on_test, plot_loss


In [ ]:
# =============================================================================
# Configuración MLflow + S3 (MinIO)
# =============================================================================

mlflow.set_tracking_uri("http://localhost:5001")
bucket_name = 'data'

s3_client = boto3.client(
    's3',
    endpoint_url=os.getenv("MLFLOW_S3_ENDPOINT_URL"),
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY")
)

def load_pickle_from_s3(key):
    obj = s3_client.get_object(Bucket=bucket_name, Key=key)
    return pickle.load(io.BytesIO(obj['Body'].read()))


In [ ]:
# =============================================================================
# Cargar datos procesados del DAG
# =============================================================================

data_by_ch = load_pickle_from_s3("processed/model_compatible/data_by_ch.pkl")

train_channels = wr.s3.read_csv(f"s3://{bucket_name}/processed/splits/train_channels.csv")["channel"].tolist()
val_channels   = wr.s3.read_csv(f"s3://{bucket_name}/processed/splits/val_channels.csv")["channel"].tolist()
test_channels  = wr.s3.read_csv(f"s3://{bucket_name}/processed/splits/test_channels.csv")["channel"].tolist()

train_channels_aug = train_channels * 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset base para scalers
train_dataset_base = DiameterDataset(
    data_dict=data_by_ch,
    ch_list=train_channels_aug,
    window_size=21,
    min_context=21,
    max_context=121,
    fit_scaler=True,
    seed=42
)


In [ ]:
# =============================================================================
# Experimento
# =============================================================================
experiment_name = "Diameter_CNN1D_HyperSearch"
if not mlflow.get_experiment_by_name(experiment_name):
    mlflow.create_experiment(experiment_name)

mlflow.set_experiment(experiment_name)


In [ ]:
# =============================================================================
# Función objetivo para Optuna
# =============================================================================
def objective(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    nlayers = trial.suggest_int("nlayers", 2, 6)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])
    window_size = trial.suggest_int("window_size", 11, 41, step=2)
    max_context = trial.suggest_int("max_context", window_size + 20, 201, step=20)
    scheduler_factor = trial.suggest_float("scheduler_factor", 0.1, 0.9)
    scheduler_patience = trial.suggest_int("scheduler_patience", 3, 10)

    train_dataset = DiameterDataset(
        data_dict=data_by_ch,
        ch_list=train_channels_aug,
        window_size=window_size,
        min_context=window_size,
        max_context=max_context,
        feature_scaler=train_dataset_base.feature_scaler,
        target_scaler=train_dataset_base.target_scaler,
        target="center",
        data_stride=1,
        seed=42
    )
    val_dataset = DiameterDataset(
        data_dict=data_by_ch,
        ch_list=val_channels,
        window_size=window_size,
        min_context=window_size,
        max_context=max_context,
        feature_scaler=train_dataset_base.feature_scaler,
        target_scaler=train_dataset_base.target_scaler,
        target="center",
        data_stride=1
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True,
                              num_workers=0, pin_memory=True, collate_fn=custom_collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=True,
                            collate_fn=custom_collate_fn)

    model = CNN1DRegressor(input_channels=3, nlayers=nlayers, dropout=dropout).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                           factor=scheduler_factor, patience=scheduler_patience)

    patience_es = 20
    best_val_loss = float('inf')
    counter = 0
    best_state = None

    for epoch in range(cfg.epochs):
        model.train()
        for X, y, _ in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X, y, _ in val_loader:
                X, y = X.to(device), y.to(device)
                out = model(X)
                val_loss += criterion(out, y).item() * X.size(0)
        val_loss /= len(val_loader.dataset)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()
            counter = 0
        else:
            counter += 1
            if counter >= patience_es:
                break

    model.load_state_dict(best_state)
    return best_val_loss


In [ ]:
# =============================================================================
# Búsqueda de hiperparámetros
# =============================================================================
with mlflow.start_run(run_name=f"Optuna_Parent_{datetime.now().strftime('%Y%m%d_%H%M')}"):
    mlflow.log_param("n_trials", 60)

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=20)
    )
    study.optimize(objective, n_trials=60)

    best_params = study.best_params
    best_val_loss = study.best_value

    mlflow.log_metric("best_val_loss", best_val_loss)
    mlflow.log_params({f"best_{k}": v for k, v in best_params.items()})

    print("=== MEJORES PARÁMETROS ===")
    for k, v in best_params.items():
        print(f"{k}: {v}")


In [ ]:
# =============================================================================
# Entrenamiento final con los mejores hiperparámetros
# =============================================================================
with mlflow.start_run(run_name=f"Final_Model_BestParams_{datetime.now().strftime('%Y%m%d_%H%M')}", nested=True):
    final_train_ds = DiameterDataset(
        data_dict=data_by_ch,
        ch_list=train_channels_aug,
        window_size=best_params["window_size"],
        min_context=best_params["window_size"],
        max_context=best_params["max_context"],
        feature_scaler=train_dataset_base.feature_scaler,
        target_scaler=train_dataset_base.target_scaler,
        target="center",
        data_stride=1,
        seed=42
    )
    final_val_ds = DiameterDataset(
        data_dict=data_by_ch,
        ch_list=val_channels,
        window_size=best_params["window_size"],
        min_context=best_params["window_size"],
        max_context=best_params["max_context"],
        feature_scaler=train_dataset_base.feature_scaler,
        target_scaler=train_dataset_base.target_scaler,
        target="center",
        data_stride=1
    )
    final_test_ds = DiameterDataset(
        data_dict=data_by_ch,
        ch_list=test_channels,
        window_size=best_params["window_size"],
        min_context=best_params["window_size"],
        max_context=best_params["max_context"],
        feature_scaler=train_dataset_base.feature_scaler,
        target_scaler=train_dataset_base.target_scaler,
        target="center",
        data_stride=1
    )

    train_loader = DataLoader(final_train_ds, batch_size=best_params["batch_size"], shuffle=True,
                              drop_last=True, num_workers=0, pin_memory=True, collate_fn=custom_collate_fn)
    val_loader   = DataLoader(final_val_ds, batch_size=best_params["batch_size"], shuffle=False,
                              drop_last=True, collate_fn=custom_collate_fn)
    test_loader  = DataLoader(final_test_ds, batch_size=1024, shuffle=False, drop_last=False)

    model = CNN1DRegressor(input_channels=3, nlayers=best_params["nlayers"],
                           dropout=best_params["dropout"]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=best_params["lr"],
                                 weight_decay=best_params["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=best_params["scheduler_factor"],
        patience=best_params["scheduler_patience"])
    criterion = nn.MSELoss()

    train_losses, val_losses = [], []
    for epoch in range(cfg.epochs):
        model.train()
        train_loss = 0.0
        for X, y, _ in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X.size(0)
        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X, y, _ in val_loader:
                X, y = X.to(device), y.to(device)
                out = model(X)
                val_loss += criterion(out, y).item() * X.size(0)
        val_loss /= len(val_loader.dataset)
        val_losses.append(val_loss)
        scheduler.step(val_loss)

        print(f"Epoch {epoch+1}/{cfg.epochs} - Train: {train_loss:.6f} - Val: {val_loss:.6f}")

    mlflow.log_metric("final_train_loss", train_losses[-1])
    mlflow.log_metric("final_val_loss", val_losses[-1])

    # Curva de pérdida
    fig, ax = plt.subplots()
    ax.plot(train_losses, label="Train")
    ax.plot(val_losses, label="Val")
    ax.legend()
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    mlflow.log_figure(fig, "loss_curve.png")
    plt.close(fig)

    # Evaluación en test
    model.eval()
    preds_scaled, targets_scaled = [], []
    preds_orig, targets_orig = [], []

    with torch.no_grad():
        for X, y, _ in test_loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            preds_scaled.extend(out.cpu().numpy())
            targets_scaled.extend(y.cpu().numpy())

            if train_dataset_base.target_scaler:
                p_orig = train_dataset_base.target_scaler.inverse_transform(
                    np.array(out.cpu()).reshape(-1, 1)).flatten()
                t_orig = train_dataset_base.target_scaler.inverse_transform(
                    np.array(y.cpu()).reshape(-1, 1)).flatten()
                preds_orig.extend(p_orig)
                targets_orig.extend(t_orig)

    mse_scaled = mean_squared_error(targets_scaled, preds_scaled)
    rmse_scaled = np.sqrt(mse_scaled)
    mae_scaled = mean_absolute_error(targets_scaled, preds_scaled)
    r2_scaled = r2_score(targets_scaled, preds_scaled)

    mse = mean_squared_error(targets_orig, preds_orig)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(targets_orig, preds_orig)
    r2 = r2_score(targets_orig, preds_orig)

    metrics = {
        "test_mse_scaled": mse_scaled, "test_rmse_scaled": rmse_scaled,
        "test_mae_scaled": mae_scaled, "test_r2_scaled": r2_scaled,
        "test_mse": mse, "test_rmse": rmse, "test_mae": mae, "test_r2": r2
    }
    mlflow.log_metrics(metrics)

    # Log del modelo
    sample_input, _, _ = next(iter(train_loader))
    signature = infer_signature(sample_input.numpy(), model(sample_input.to(device)).cpu().numpy())

    mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="model",
        signature=signature,
        registered_model_name="diameter_cnn1d_prod",
        metadata={"model_version": 1, "trained_on": datetime.now().isoformat()}
    )

    model_uri = mlflow.get_artifact_uri("model")
    print(f"Modelo final guardado en: {model_uri}")


In [ ]:
# =============================================================================
# Registrar modelo en Model Registry con alias "champion"
# =============================================================================
from mlflow import MlflowClient

client = MlflowClient()

registered_model_name = "diameter_cnn1d_prod"
description = "CNN1D Regressor con contexto variable para MeanRateOutDiam (ISI 2024)"

try:
    client.create_registered_model(name=registered_model_name, description=description)
except:
    client.update_registered_model(name=registered_model_name, description=description)

tags = {
    "model_type": "CNN1DRegressor",
    "framework": "PyTorch",
    "task": "regression",
    "target": "MeanRateOutDiam",
    "window_size": best_params["window_size"],
    "max_context": best_params["max_context"],
    "nlayers": best_params["nlayers"],
    "dropout": best_params["dropout"],
    "lr": best_params["lr"],
    "batch_size": best_params["batch_size"],
    "weight_decay": best_params["weight_decay"],
    "scheduler_factor": best_params["scheduler_factor"],
    "scheduler_patience": best_params["scheduler_patience"],
    "train_augmentation": 3,
    "data_stride": 1,
    "seed": cfg.seed,
    "test_rmse": rmse,
    "test_mae": mae,
    "test_r2": r2,
    "test_rmse_scaled": rmse_scaled,
    "final_val_loss": val_losses[-1],
}

current_run = mlflow.active_run()
run_id = current_run.info.run_id
source = f"runs:/{run_id}/model"

model_version = client.create_model_version(
    name=registered_model_name,
    source=source,
    run_id=run_id,
    tags=tags
)

client.set_registered_model_alias(
    name=registered_model_name,
    alias="champion",
    version=model_version.version
)

print(f"Nueva versión creada: {model_version.version} → alias 'champion'")

In [ ]:
import joblib
joblib.dump(train_dataset_base.feature_scaler, "/tmp/feature_scaler.pkl")
joblib.dump(train_dataset_base.target_scaler, "/tmp/target_scaler.pkl")
mlflow.log_artifact("/tmp/feature_scaler.pkl")
mlflow.log_artifact("/tmp/target_scaler.pkl")